# Etap 1 - Pozyskanie danych z cBioPortal API

**Projekt:** Analiza przeżywalności pacjentów z glejakami w kohorcie TCGA  
**Autor:** Anna Zimniewicz  
**Data:** 18 maja 2026

## Cel notebooka

Pobranie surowych danych klinicznych i molekularnych dla dwóch zbiorów TCGA z cBioPortal:
- **TCGA-GBM** (Glioblastoma Multiforme, Firehose Legacy) - 619 próbek
- **TCGA-LGG** (Brain Lower Grade Glioma, Firehose Legacy) - 530 próbek

Dane zostaną zapisane jako surowe pliki CSV w folderze `data/raw/`.

## Plan

1. Konfiguracja: import bibliotek, ustawienie bazowego URL API
2. Sprawdzenie statusu serwera cBioPortal (`/api/health`)
3. Pobranie listy wszystkich studies i znalezienie `studyId` dla naszych dwóch kohort
4. Pobranie danych klinicznych dla obu studies
5. Pobranie mutacji dla genów IDH1, IDH2
6. Zapis surowych danych do `../data/raw/`

## Źródło danych

- cBioPortal: https://www.cbioportal.org
- Dokumentacja API: https://www.cbioportal.org/api/swagger-ui/index.html

---

## Sekcja 1 — Konfiguracja

Importy bibliotek, definicja stałych projektu (URL bazowy API, identyfikatory studies, ścieżka do surowych danych).

In [1]:
# Import bibliotek potrzebnych do pobierania i obróbki danych
import requests              # wysyłanie zapytań HTTP do API
import pandas as pd          # praca z danymi tabelarycznymi 
from pathlib import Path     # nowoczesna obsługa ścieżek plików 
from datetime import datetime  # do oznaczania daty pobrania danych

# Sprawdzamy, czy biblioteki się załadowały
print("Wszystkie biblioteki zaimportowane poprawnie.")
print(f"pandas version: {pd.__version__}")
print(f"requests version: {requests.__version__}")

Wszystkie biblioteki zaimportowane poprawnie.
pandas version: 3.0.2
requests version: 2.33.1


In [2]:
# === KONFIGURACJA ===

# Bazowy URL cBioPortal API
# Wszystkie endpointy budujemy doklejając kolejne fragmenty do tego URL-a
BASE_URL = "https://www.cbioportal.org/api"

# Identyfikatory dwóch studies, które nas interesują
# (te ID znajdziemy za chwilę przez API - na razie wpisujemy je z głowy na podstawie znajomości cBioPortal)
STUDY_IDS = {
    "GBM": "gbm_tcga",       # Glioblastoma Multiforme (TCGA, Firehose Legacy)
    "LGG": "lgg_tcga",       # Brain Lower Grade Glioma (TCGA, Firehose Legacy)
}

# Ścieżka do folderu na surowe dane
# Path("..") oznacza "folder wyżej" - bo notebook jest w notebooks/, a data jest w głównym folderze projektu
RAW_DATA_DIR = Path("..") / "data" / "raw"

# Upewniamy się, że folder istnieje (jeśli nie - utwórz)
# parents=True: utwórz też foldery nadrzędne, jeśli nie istnieją
# exist_ok=True: nie rzucaj błędu, jeśli folder już jest
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Data pobrania - przyda się do nazw plików i dokumentacji
DOWNLOAD_DATE = datetime.now().strftime("%Y-%m-%d")

# Wypisujemy konfigurację dla kontroli
print(f"Base URL: {BASE_URL}")
print(f"Studies: {STUDY_IDS}")
print(f"Folder docelowy: {RAW_DATA_DIR.resolve()}")  # .resolve() pokazuje pełną ścieżkę
print(f"Data pobrania: {DOWNLOAD_DATE}")

Base URL: https://www.cbioportal.org/api
Studies: {'GBM': 'gbm_tcga', 'LGG': 'lgg_tcga'}
Folder docelowy: C:\Users\anazi\Desktop\praca_dyplomowa\data\raw
Data pobrania: 2026-06-14


---

## Sekcja 2 — Weryfikacja serwera i studies

Sprawdzamy, czy:
- serwer cBioPortal odpowiada poprawnie (`/api/health`),
- studies `gbm_tcga` i `lgg_tcga` istnieją i są to faktycznie TCGA Firehose Legacy,
- liczby pacjentów odpowiadają wartościom widocznym na stronie webowej cBioPortal.

**Uwaga diagnostyczna:** podczas weryfikacji wykryto błąd w polu `allSampleCount` w metadanych API (zwraca `1` zamiast prawidłowej liczby próbek). Bug nie wpływa na dalszą pracę — realną wielkość kohorty ustalamy na podstawie endpointu `/patients`.

In [3]:
# === KROK 1: Sprawdzenie statusu serwera cBioPortal ===

# Budujemy URL endpointu przez sklejenie bazowego URL-a z konkretną ścieżką
health_url = f"{BASE_URL}/health"

# Wysyłamy zapytanie GET
# requests.get() zwraca obiekt Response zawierający odpowiedź serwera
response = requests.get(health_url)

# Sprawdzamy, czy zapytanie się powiodło
# raise_for_status() rzuca wyjątek, jeśli status_code nie jest w zakresie 200-299
response.raise_for_status()

# Wyświetlamy informacje diagnostyczne
print(f"URL zapytania: {health_url}")
print(f"Status code: {response.status_code}")
print(f"Czas odpowiedzi: {response.elapsed.total_seconds()} s")
print(f"Treść odpowiedzi: {response.text}")

URL zapytania: https://www.cbioportal.org/api/health
Status code: 200
Czas odpowiedzi: 0.726909 s
Treść odpowiedzi: {"status":"UP"}


In [4]:
# === KROK 2: Pobranie listy wszystkich studies w cBioPortal ===

# Endpoint /studies zwraca listę wszystkich dostępnych badań w cBioPortal
# Dodajemy parametr projection=SUMMARY - dostaniemy skrócone metadane (bez wszystkich detali)

studies_url = f"{BASE_URL}/studies" 
params = {
    "projection" : "SUMMARY", 
    "pageSize" : 10_000_000,
    "pageNumber" : 0,
    "direction" : "ASC",
}

response = requests.get(studies_url, params=params)
response.raise_for_status()

studies_data = response.json()

print(f"Liczba studies w cBioPortal: {len(studies_data)}")
print(f"Typ obiektu: {type(studies_data)}")
print(f"Typ pierwszego elementu: {type(studies_data[0])}")
print(f"\nKlucze w pierwszym elemencie:")
print(list(studies_data[0].keys()))



Liczba studies w cBioPortal: 535
Typ obiektu: <class 'list'>
Typ pierwszego elementu: <class 'dict'>

Klucze w pierwszym elemencie:
['studyId', 'cancerTypeId', 'name', 'description', 'publicStudy', 'pmid', 'citation', 'groups', 'status', 'importDate', 'allSampleCount', 'referenceGenome', 'cancerType', 'readPermission', 'resourceCounts']


In [5]:
# === KROK 3: Konwersja na DataFrame i filtrowanie do naszych studies ===
# pandas potrafi zamienić listę słowników bezpośrednio w DataFrame
# Każdy słownik → jeden wiersz, klucze słownika → nazwy kolumn

studies_df = pd.DataFrame(studies_data)

print(f"Wymiary tabeli: {studies_df.shape}") #.shape zwraca (liczba_wierszy, liczba_kolumn)
print(f"Kolumny: {list(studies_df.columns)}")

print("\nTrzy pierwsze studies (wybrane kolumny):")
display(studies_df[["studyId", "name", "cancerTypeId", "allSampleCount"]].head(3)) #.head(3) zwraca 3 pierwsze wiersze 


Wymiary tabeli: (535, 15)
Kolumny: ['studyId', 'cancerTypeId', 'name', 'description', 'publicStudy', 'pmid', 'citation', 'groups', 'status', 'importDate', 'allSampleCount', 'referenceGenome', 'cancerType', 'readPermission', 'resourceCounts']

Trzy pierwsze studies (wybrane kolumny):


,studyId,name,cancerTypeId,allSampleCount
0,acbc_mskcc_2015,"Adenoid Cystic Carcinoma of the Breast (MSK, J...",acbc,12
1,acc_tcga,"Adrenocortical Carcinoma (TCGA, Firehose Legacy)",acc,92
2,acyc_mskcc_2013,"Adenoid Cystic Carcinoma (MSK, Nat Genet 2013)",acyc,60


Wyjaśnienie: te trzy wiersze to nowo dodane studies w stanie roboczym ("placeholdery" z 1 testową próbką). Sortowanie alfabetyczne wrzuciło je na początek. Nasze GBM (619 próbek) i LGG (530 próbek) są gdzieś dalej w tabeli.

In [6]:
# === KROK 4: Filtrowanie do naszych dwóch studies (GBM + LGG Firehose Legacy) ===

# Lista ID, których szukamy (bierzemy je ze słownika STUDY_IDS)
target_ids = list(STUDY_IDS.values())  # ['gbm_tcga', 'lgg_tcga'] #.values zwraca wartości słownika, list zamienia do na listę
print(f"Szukamy studies o ID: {target_ids}")

# Filtrowanie DataFrame metodą .isin()
# Tworzymy warunek logiczny: które wiersze mają studyId w naszej liście?
#Weź kolumnę studyId z DataFrame. 
#Dla każdego wiersza sprawdź, czy wartość w tej kolumnie znajduje się w liście target_ids. 
#Zwróć kolumnę z True/False.
mask = studies_df["studyId"].isin(target_ids)

our_studies = studies_df[mask]

print(f"\nZnaleziono {len(our_studies)} pasujących studies:")
display(our_studies[["studyId", "name", "cancerTypeId", "allSampleCount", "referenceGenome"]])

Szukamy studies o ID: ['gbm_tcga', 'lgg_tcga']

Znaleziono 2 pasujących studies:


,studyId,name,cancerTypeId,allSampleCount,referenceGenome
118,gbm_tcga,"Glioblastoma Multiforme (TCGA, Firehose Legacy)",difg,619,hg19
160,lgg_tcga,"Brain Lower Grade Glioma (TCGA, Firehose Legacy)",difg,530,hg19


## Sekcja 3 — Eksploracja kohorty

Pobieramy szczegółowe metadane studies (`projection=DETAILED`) oraz listę pacjentów dla obu kohort. Ustalamy realną wielkość kohorty:

- **TCGA-GBM:** 606 pacjentów
- **TCGA-LGG:** 516 pacjentów
- **Razem:** 1122 pacjentów


In [7]:
# === DIAGNOSTYKA: Sprawdzenie metadanych dla naszych studies z DETAILED projection ===

# Powtarzamy zapytanie, ale tym razem z projection=DETAILED
# Powinniśmy dostać pełne metadane, w tym prawdziwy allSampleCount
params_detailed = {
    "projection": "DETAILED",
    "pageSize": 10_000_000,
    "pageNumber": 0,
    "direction": "ASC",
}

response = requests.get(f"{BASE_URL}/studies", params=params_detailed)
response.raise_for_status()
studies_detailed = pd.DataFrame(response.json())

print(f"Liczba kolumn z DETAILED: {len(studies_detailed.columns)}")
print(f"Kolumny: {list(studies_detailed.columns)}")

# Filtrujemy do naszych dwóch studies
our_studies_detailed = studies_detailed[studies_detailed["studyId"].isin(target_ids)]

print("\nNasze studies z DETAILED projection:")
display(our_studies_detailed)

Liczba kolumn z DETAILED: 27
Kolumny: ['studyId', 'cancerTypeId', 'name', 'description', 'publicStudy', 'pmid', 'citation', 'groups', 'status', 'importDate', 'allSampleCount', 'sequencedSampleCount', 'cnaSampleCount', 'mrnaRnaSeqSampleCount', 'mrnaRnaSeqV2SampleCount', 'mrnaMicroarraySampleCount', 'miRnaSampleCount', 'methylationHm27SampleCount', 'rppaSampleCount', 'massSpectrometrySampleCount', 'completeSampleCount', 'referenceGenome', 'treatmentCount', 'structuralVariantCount', 'cancerType', 'readPermission', 'resourceCounts']

Nasze studies z DETAILED projection:


,studyId,cancerTypeId,name,description,publicStudy,pmid,citation,groups,status,importDate,...,methylationHm27SampleCount,rppaSampleCount,massSpectrometrySampleCount,completeSampleCount,referenceGenome,treatmentCount,structuralVariantCount,cancerType,readPermission,resourceCounts
243,gbm_tcga,difg,"Glioblastoma Multiforme (TCGA, Firehose Legacy)",TCGA Glioblastoma Multiforme. Source data from...,True,NaN,NaN,PUBLIC,0,2026-01-07 12:59:53,...,285,244,0,136,hg19,0,0,"{'id': 'difg', 'name': 'Diffuse Glioma', 'dedi...",True,[]
413,lgg_tcga,difg,"Brain Lower Grade Glioma (TCGA, Firehose Legacy)",TCGA Brain Lower Grade Glioma. Source data fro...,True,NaN,NaN,PUBLIC,0,2026-01-07 15:31:49,...,0,435,0,283,hg19,0,0,"{'id': 'difg', 'name': 'Diffuse Glioma', 'dedi...",True,[]


## Uwaga diagnostyczna - bug w cBioPortal API

Endpoint `/studies` zwraca `allSampleCount: 1` dla obu naszych studies (`gbm_tcga`, `lgg_tcga`), 
zarówno z `projection=SUMMARY` jak i `DETAILED`. To jest **bug w metadanych cBioPortal** - na stronie 
webowej widać poprawne liczby (619 dla GBM, 530 dla LGG), ale licznik w API się rozjechał.

**Inne pola w DETAILED zawierają prawdziwe liczby per typ danych:**
- `sequencedSampleCount`: 290 (GBM), 286 (LGG) - próbki z danymi mutacji
- `cnaSampleCount`: 577 (GBM), 513 (LGG) - próbki z danymi CNA

**Realna wielkość kohorty zostanie ustalona** na podstawie liczby unikalnych pacjentów 
w pobranych danych klinicznych (krok 5).

In [8]:
# === KROK 5: Pobranie listy pacjentów dla obu studies ===

# Endpoint /studies/{studyId}/patients zwraca listę wszystkich pacjentów w danym studium
# Robimy to w pętli dla obu naszych studies (GBM i LGG)

patients_per_study = {}  # tu będziemy zbierać wyniki

for label, study_id in STUDY_IDS.items():
    # label = "GBM" lub "LGG", study_id = "gbm_tcga" lub "lgg_tcga"
    
    # Budujemy URL z konkretnym study_id
    url = f"{BASE_URL}/studies/{study_id}/patients"
    
    # Wysyłamy zapytanie z dużym pageSize, żeby dostać wszystko naraz
    params = {"pageSize": 100_000, "pageNumber": 0}
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    # Parsujemy odpowiedź i zamieniamy na DataFrame
    patients = pd.DataFrame(response.json())
    
    # Zapisujemy do słownika pod kluczem "GBM" / "LGG"
    patients_per_study[label] = patients
    
    print(f"{label} ({study_id}): {len(patients)} pacjentów")
    
print(f"\nŁącznie pacjentów: {sum(len(df) for df in patients_per_study.values())}")
print("\nKolumny w DataFrame pacjentów GBM:")
print(list(patients_per_study["GBM"].columns))
print("\nPierwsze 3 wiersze GBM:")
display(patients_per_study["GBM"].head(3))

GBM (gbm_tcga): 606 pacjentów
LGG (lgg_tcga): 516 pacjentów

Łącznie pacjentów: 1122

Kolumny w DataFrame pacjentów GBM:
['uniquePatientKey', 'patientId', 'studyId']

Pierwsze 3 wiersze GBM:


,uniquePatientKey,patientId,studyId
0,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga
1,VENHQS0wMi0wMDAzOmdibV90Y2dh,TCGA-02-0003,gbm_tcga
2,VENHQS0wMi0wMDA2OmdibV90Y2dh,TCGA-02-0006,gbm_tcga


---

## Sekcja 4 — Diagnostyka dostępności statusu MGMT i IDH

**Procedura diagnostyczna:**
1. Pobranie atrybutów klinicznych na poziomie pacjenta (`PATIENT`) dla `gbm_tcga` i `lgg_tcga`.
2. Sprawdzenie atrybutów na poziomie próbki (`SAMPLE`).
3. Weryfikacja alternatywnych wersji studies (PanCancer Atlas, GDC 2025, studies publikacyjne).
4. Sprawdzenie surowych plików danych w repozytorium cBioPortal DataHub na GitHubie.

**Konkluzja:** żadne ze sprawdzonych studies cBioPortal nie udostępnia `MGMT_STATUS` i `IDH_STATUS` jako gotowych atrybutów na poziomie pacjenta. Konieczna jest decyzja metodologiczna (patrz Sekcja 5).

In [9]:
# === KROK 6: Pobranie danych klinicznych w formacie LONG ===

# Słownik, w którym zbierzemy dane kliniczne per studium
clinical_long_per_study = {}

for label, study_id in STUDY_IDS.items():
    # Endpoint zwraca wszystkie dane kliniczne dla pacjentów w studium
    url = f"{BASE_URL}/studies/{study_id}/clinical-data"
    
    # clinicalDataType=PATIENT - chcemy atrybuty pacjenta (nie próbki)
    # projection=SUMMARY - mniej pól, ale wszystko, czego nam trzeba
    params = {
        "clinicalDataType": "PATIENT",
        "projection": "SUMMARY",
        "pageSize": 1_000_000,
        "pageNumber": 0,
    }
    
    print(f"Pobieram dane kliniczne dla {label} ({study_id})...")
    response = requests.get(url, params=params)
    response.raise_for_status()
    
    # Zamieniamy odpowiedź na DataFrame
    clinical_long = pd.DataFrame(response.json())
    clinical_long_per_study[label] = clinical_long
    
    print(f"  Pobrano {len(clinical_long):,} wierszy (long format)")
    print(f"  Liczba unikalnych pacjentów: {clinical_long['patientId'].nunique()}")
    print(f"  Liczba unikalnych atrybutów: {clinical_long['clinicalAttributeId'].nunique()}")
    print()

# Pokaż pierwsze 10 wierszy GBM dla wglądu w surową strukturę long
print("Pierwsze 10 wierszy danych klinicznych GBM (format LONG):")
display(clinical_long_per_study["GBM"].head(10))

Pobieram dane kliniczne dla GBM (gbm_tcga)...
  Pobrano 14,830 wierszy (long format)
  Liczba unikalnych pacjentów: 606
  Liczba unikalnych atrybutów: 34

Pobieram dane kliniczne dla LGG (lgg_tcga)...
  Pobrano 25,322 wierszy (long format)
  Liczba unikalnych pacjentów: 516
  Liczba unikalnych atrybutów: 65

Pierwsze 10 wierszy danych klinicznych GBM (format LONG):


,uniquePatientKey,patientId,studyId,clinicalAttributeId,value
0,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,AGE,44
1,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS,0
2,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,DFS_MONTHS,4.5
3,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,DFS_STATUS,1:Recurred/Progressed
4,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,ETHNICITY,NOT HISPANIC OR LATINO
5,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,FORM_COMPLETION_DATE,12/16/08
6,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,HISTOLOGICAL_DIAGNOSIS,Untreated primary (de novo) GBM
7,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,HISTORY_LGG_DX_OF_BRAIN_TISSUE,NO
8,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,HISTORY_NEOADJUVANT_TRTYN,Yes
9,VENHQS0wMi0wMDAxOmdibV90Y2dh,TCGA-02-0001,gbm_tcga,ICD_10,C71.9


In [10]:
# === KROK 7: Eksploracja - jakie atrybuty kliniczne są dostępne ===

# Pobieramy unikalne nazwy atrybutów dla każdego studium
attrs_gbm = set(clinical_long_per_study["GBM"]["clinicalAttributeId"].unique())
attrs_lgg = set(clinical_long_per_study["LGG"]["clinicalAttributeId"].unique())

# Atrybuty wspólne (są w obu studies)
common_attrs = attrs_gbm & attrs_lgg  # operator & to "część wspólna" zbiorów

# Atrybuty tylko w jednym ze studies
only_gbm = attrs_gbm - attrs_lgg
only_lgg = attrs_lgg - attrs_gbm

print(f"Atrybuty wspólne dla obu studies: {len(common_attrs)}")
print(f"Atrybuty tylko w GBM: {len(only_gbm)}")
print(f"Atrybuty tylko w LGG: {len(only_lgg)}")

print("\n=== Atrybuty wspólne dla obu studies (alfabetycznie) ===")
for attr in sorted(common_attrs):
    print(f"  {attr}")

print("\n=== Atrybuty tylko w LGG ===")
for attr in sorted(only_lgg):
    print(f"  {attr}")

print("\n=== Atrybuty tylko w GBM ===")
for attr in sorted(only_gbm):
    print(f"  {attr}")

Atrybuty wspólne dla obu studies: 31
Atrybuty tylko w GBM: 3
Atrybuty tylko w LGG: 34

=== Atrybuty wspólne dla obu studies (alfabetycznie) ===
  AGE
  DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS
  DFS_MONTHS
  DFS_STATUS
  ECOG_SCORE
  ETHNICITY
  FORM_COMPLETION_DATE
  HISTOLOGICAL_DIAGNOSIS
  HISTORY_NEOADJUVANT_TRTYN
  HISTORY_OTHER_MALIGNANCY
  ICD_10
  ICD_O_3_HISTOLOGY
  ICD_O_3_SITE
  INFORMED_CONSENT_VERIFIED
  INITIAL_PATHOLOGIC_DX_YEAR
  KARNOFSKY_PERFORMANCE_SCORE
  NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT
  OS_MONTHS
  OS_STATUS
  OTHER_PATIENT_ID
  PERFORMANCE_STATUS_TIMING
  PROSPECTIVE_COLLECTION
  RACE
  RADIATION_TREATMENT_ADJUVANT
  RETROSPECTIVE_COLLECTION
  SAMPLE_COUNT
  SEX
  SITE_OF_TUMOR_TISSUE
  TISSUE_SOURCE_SITE
  TREATMENT_OUTCOME_FIRST_COURSE
  TUMOR_STATUS

=== Atrybuty tylko w LGG ===
  ANIMAL_INSECT_ALLERGY_AGE
  ANIMAL_INSECT_ALLERGY_HIST
  ASTHMA_ECZEMA_ALLERGY_FIRST_DIAGNOSIS
  ASTHMA_HISTORY
  ECZEMA_HISTORY
  FAMILY_HISTORY_OF_CANCER
  FAMILY_HISTORY_OF_PR

In [11]:
# === DIAGNOZA: sprawdzenie atrybutów na poziomie SAMPLE (próbki) ===

# Powtarzamy zapytanie, ale tym razem dla danych na poziomie próbki, nie pacjenta
url = f"{BASE_URL}/studies/gbm_tcga/clinical-data"
params = {
    "clinicalDataType": "SAMPLE",   # ← TO jest jedyna zmiana
    "projection": "SUMMARY",
    "pageSize": 1_000_000,
    "pageNumber": 0,
}

response = requests.get(url, params=params)
response.raise_for_status()
sample_clinical_gbm = pd.DataFrame(response.json())

print(f"Pobrano {len(sample_clinical_gbm):,} wierszy danych SAMPLE dla GBM")
print(f"Liczba unikalnych atrybutów SAMPLE: {sample_clinical_gbm['clinicalAttributeId'].nunique()}")

# Pokażemy wszystkie atrybuty SAMPLE
sample_attrs_gbm = sorted(sample_clinical_gbm["clinicalAttributeId"].unique())
print("\n=== Wszystkie atrybuty na poziomie SAMPLE dla GBM ===")
for attr in sample_attrs_gbm:
    # Podświetlamy te, które brzmią dla nas istotnie
    is_key = any(keyword in attr.upper() for keyword in ["MGMT", "IDH", "METHYL", "MUTATION"])
    marker = "  ⭐ " if is_key else "     "
    print(f"{marker}{attr}")

Pobrano 9,506 wierszy danych SAMPLE dla GBM
Liczba unikalnych atrybutów SAMPLE: 20

=== Wszystkie atrybuty na poziomie SAMPLE dla GBM ===
     CANCER_TYPE
     CANCER_TYPE_DETAILED
     DAYS_TO_COLLECTION
     FRACTION_GENOME_ALTERED
     IS_FFPE
     LONGEST_DIMENSION
  ⭐ MUTATION_COUNT
     OCT_EMBEDDED
     ONCOTREE_CODE
     OTHER_SAMPLE_ID
     PATHOLOGY_REPORT_FILE_NAME
     PATHOLOGY_REPORT_UUID
     SAMPLE_INITIAL_WEIGHT
     SAMPLE_TYPE
     SAMPLE_TYPE_ID
     SHORTEST_DIMENSION
     SOMATIC_STATUS
     SPECIMEN_SECOND_LONGEST_DIMENSION
     TMB_NONSYNONYMOUS
     VIAL_NUMBER


In [12]:
# === SYSTEMATYCZNY PRZEGLĄD: jakie studies GBM/LGG są dostępne w cBioPortal ===

# Mamy już studies_df z poprzedniego kroku (lista wszystkich 535 studies)
# Filtrujemy do tych, które dotyczą glejaków

# Szukamy po nazwie - "glioblastoma" lub "glioma" lub "glial" w nazwie (case-insensitive)
mask = studies_df["name"].str.contains("glio", case=False, na=False)
glioma_studies = studies_df[mask][["studyId", "name", "allSampleCount"]]

print(f"Studies dotyczące glejaków: {len(glioma_studies)}")
display(glioma_studies)

Studies dotyczące glejaków: 23


,studyId,name,allSampleCount
100,difg_glass_2019,"Diffuse Glioma (GLASS Consortium, Nature 2019)",444
118,gbm_tcga,"Glioblastoma Multiforme (TCGA, Firehose Legacy)",619
122,gbm_tcga_pub2013,"Glioblastoma (TCGA, Cell 2013)",577
124,glioma_mskcc_2019,"Glioma (MSK, Clin Cancer Res 2019)",1004
125,glioma_msk_2018,"Glioma (MSK, Nature 2019)",91
128,gbm_columbia_2019,"Glioblastoma (Columbia, Nat Med. 2019)",42
130,gbm_cptac_2021,"Glioblastoma (CPTAC, Cell 2021)",99
149,hnsc_a5consortium_2025,Hereditary SDHB-Mutant Pheochromocytomas and P...,94
160,lgg_tcga,"Brain Lower Grade Glioma (TCGA, Firehose Legacy)",530
163,lgg_ucsf_2014,"Low-Grade Gliomas (UCSF, Science 2014)",61


In [13]:
# === DIAGNOSTYKA: PanCancer Atlas studies - sprawdzamy atrybuty PATIENT ===

pancancer_studies = [
    "gbm_tcga_pan_can_atlas_2018",
    "lgg_tcga_pan_can_atlas_2018",
    "difg_tcga_gdc",        # Diffuse Glioma - może mieć już gotowe statusy
    "gbm_tcga_pub2013",     # Brennan 2013
    "gbm_tcga_pub",         # TCGA Nature 2008
]

# Słowa kluczowe szerzej - łapiemy więcej wariantów
keywords = ["MGMT", "IDH", "METHYL", "MUTATION_STATUS", "1P", "19Q", "G_CIMP", "GRADE", "SUBTYPE"]

for study_id in pancancer_studies:
    print(f"\n=== {study_id} ===")
    
    url = f"{BASE_URL}/studies/{study_id}/clinical-data"
    params = {
        "clinicalDataType": "PATIENT",
        "projection": "SUMMARY",
        "pageSize": 1_000_000,
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        df = pd.DataFrame(response.json())
        
        n_patients = df["patientId"].nunique()
        attrs = sorted(df["clinicalAttributeId"].unique())
        relevant = [a for a in attrs if any(kw in a.upper() for kw in keywords)]
        
        print(f"  Pacjentów: {n_patients}, atrybutów PATIENT łącznie: {len(attrs)}")
        if relevant:
            print(f"  ⭐ Istotne atrybuty:")
            for a in relevant:
                print(f"     {a}")
        else:
            print(f"  ❌ Brak interesujących atrybutów")
    
    except requests.exceptions.HTTPError as e:
        print(f"  ❌ Błąd HTTP: {e.response.status_code}")


=== gbm_tcga_pan_can_atlas_2018 ===
  Pacjentów: 585, atrybutów PATIENT łącznie: 34
  ⭐ Istotne atrybuty:
     SUBTYPE

=== lgg_tcga_pan_can_atlas_2018 ===
  Pacjentów: 514, atrybutów PATIENT łącznie: 34
  ⭐ Istotne atrybuty:
     SUBTYPE

=== difg_tcga_gdc ===
  Pacjentów: 516, atrybutów PATIENT łącznie: 26
  ❌ Brak interesujących atrybutów

=== gbm_tcga_pub2013 ===
  Pacjentów: 577, atrybutów PATIENT łącznie: 8
  ❌ Brak interesujących atrybutów

=== gbm_tcga_pub ===
  Pacjentów: 206, atrybutów PATIENT łącznie: 9
  ❌ Brak interesujących atrybutów


In [14]:
# === DIAGNOSTYKA: pobranie surowego pliku data_clinical_patient.txt z GitHub DataHub ===

# Bezpośredni URL do surowego pliku z repo cBioPortal/datahub
# media.githubusercontent.com obsługuje Git LFS (large file storage) - duże pliki
url = "https://media.githubusercontent.com/media/cBioPortal/datahub/master/public/gbm_tcga/data_clinical_patient.txt"

print(f"Pobieram plik z: {url}")
response = requests.get(url)
print(f"Status: {response.status_code}")
print(f"Rozmiar pliku: {len(response.text):,} znaków")
print(f"\nPierwsze 2000 znaków pliku:\n")
print(response.text[:2000])

Pobieram plik z: https://media.githubusercontent.com/media/cBioPortal/datahub/master/public/gbm_tcga/data_clinical_patient.txt
Status: 200
Rozmiar pliku: 278,508 znaków

Pierwsze 2000 znaków pliku:

#Other Patient ID	Patient Identifier	Form completion date	History lgg dx of brain tissue	Tissue Prospective Collection Indicator	Tissue Retrospective Collection Indicator	Sex	Race Category	Ethnicity Category	Prior Cancer Diagnosis Occurence	Neoadjuvant Therapy Type Administered Prior To Resection Text	Year Cancer Initial Diagnosis	First Pathologic Diagnosis Biospecimen Acquisition Method Type	First Pathologic Diagnosis Biospecimen Acquisition Other Method Type	Person Neoplasm Status	Karnofsky Performance Score	Performance Status	Performance Status Assessment Timepoint Category	Did patient start adjuvant postoperative radiotherapy?	Adjuvant Postoperative Pharmaceutical Therapy Administered Indicator	Primary Therapy Outcome Success Type	New Neoplasm Event Post Initial Therapy Indicator	Diagno

---

## Sekcja 5 — Decyzja metodologiczna i plan dalszy

### Podjęta decyzja

Status MGMT i IDH dla pacjentów TCGA-GBM i TCGA-LGG zostaną zaczerpnięte z **Supplementary Materials publikacji Ceccarelli et al. 2016** ("Molecular Profiling Reveals Biologically Discrete Subsets and Pathways of Progression in Diffuse Glioma", *Cell*, 164:550-563). Publikacja zawiera autorytatywną klasyfikację molekularną dla ~1100 pacjentów TCGA, używaną w setkach kolejnych prac.

### Dodatkowa walidacja

Status IDH zostanie dodatkowo zwalidowany przez porównanie z mutacjami genów IDH1 i IDH2 pobranymi z endpointu `/mutations` cBioPortal. Zgodność oczekiwana ~95%+.

### Plan na następną sesję

1. Pobranie danych klinicznych (`/clinical-data`) dla obu studies — wiek, płeć, OS_MONTHS, OS_STATUS, leczenie itd.
2. Pobranie Supplementary Table z publikacji Ceccarelli 2016 (gotowe statusy MGMT, IDH, 1p/19q codeletion, SUBTYPE).
3. Pobranie mutacji IDH1/IDH2 — walidacja statusu IDH.
4. Zapis surowych plików do `data/raw/`.
5. Aktualizacja `README.md` z opisem pobranych danych i datą.

### Status Etapu 1

- ✅ Weryfikacja studies i kohorty
- ✅ Diagnostyka dostępności kluczowych zmiennych molekularnych
- ⏳ Pobranie i zapis surowych danych klinicznych
- ⏳ Pobranie statusów MGMT/IDH (Ceccarelli 2016)
- ⏳ Walidacja statusu IDH z mutacjami

## 6. Wczytanie tabeli klinicznej z Ceccarelli 2016

Pełne dane kliniczne + status molekularny (IDH, MGMT, 1p/19q codeletion) dla całej kohorty 1 122 pacjentów (GBM + LGG) pochodzą z materiałów suplementarnych:

> Ceccarelli M, Barthel FP, Malta TM, et al. *Molecular Profiling Reveals Biologically Discrete Subsets and Pathways of Progression in Diffuse Glioma.* Cell. 2016;164(3):550-563.

Plik `ceccarelli_2016_table_s1.xlsx` zawiera 3 arkusze; interesuje nas tylko **S1A** (TCGA discovery dataset).


In [16]:
# Ścieżka do pliku - korzystamy z RAW_DATA_DIR zdefiniowanego wcześniej
ceccarelli_path = RAW_DATA_DIR / "ceccarelli_2016_table_s1.xlsx"

# Wczytanie arkusza S1A
# UWAGA: w pliku Excela pierwszy wiersz (indeks 0) to "super-nagłówki" (kategorie kolumn typu "Clinical Data"),
# a faktyczne nazwy kolumn są w drugim wierszu (indeks 1). Dlatego header=1.
ceccarelli = pd.read_excel(
    ceccarelli_path,
    sheet_name="S1A. TCGA discovery dataset",
    header=1
)

# Podgląd rozmiaru tabeli
print(f"Liczba wierszy:  {len(ceccarelli)}")
print(f"Liczba kolumn:   {len(ceccarelli.columns)}")

Liczba wierszy:  1122
Liczba kolumn:   51


In [18]:
# Pełna lista kolumn 
print("Wszystkie kolumny w tabeli:")
for i, col in enumerate(ceccarelli.columns):
    print(f"  {i:2d}. {col}")
    

Wszystkie kolumny w tabeli:
   0. Case
   1. Tissue source site
   2. Study
   3. BCR
   4. Whole exome
   5. Whole genome
   6. RNAseq
   7. SNP6
   8. U133a
   9. HM450
  10. HM27
  11. RPPA
  12. Histology
  13. Grade
  14. Age (years at diagnosis)
  15. Gender
  16. Survival (months)
  17. Vital status (1=dead)
  18. Karnofsky Performance Score
  19. Mutation Count
  20. Percent aneuploidy
  21. IDH status
  22. 1p/19q codeletion
  23. IDH/codel subtype
  24. MGMT promoter status
  25. Chr 7 gain/Chr 10 loss
  26. Chr 19/20 co-gain
  27. TERT promoter status
  28. TERT expression (log2)
  29. TERT expression status
  30. ATRX status
  31. DAXX status
  32. Telomere Maintenance
  33. BRAF V600E status
  34. BRAF-KIAA1549 fusion
  35. ABSOLUTE purity
  36. ABSOLUTE ploidy
  37. ESTIMATE stromal score
  38. ESTIMATE immune score
  39. ESTIMATE combined score
  40. Original Subtype
  41. Transcriptome Subtype
  42. Pan-Glioma RNA Expression Cluster
  43. IDH-specific RNA Expression Clu

In [19]:
# Podgląd pierwszych 3 wierszy 
kluczowe_kolumny = [
    'Case',
    'Study',
    'Histology',
    'Grade',
    'Age (years at diagnosis)',
    'Gender',
    'Survival (months)',
    'Vital status (1=dead)',
    'IDH status',
    'MGMT promoter status',
    '1p/19q codeletion',
    'IDH/codel subtype',
]

ceccarelli[kluczowe_kolumny].head(3)

,Case,Study,Histology,Grade,Age (years at diagnosis),Gender,Survival (months),Vital status (1=dead),IDH status,MGMT promoter status,1p/19q codeletion,IDH/codel subtype
0,TCGA-CS-4938,Brain Lower Grade Glioma,astrocytoma,G2,31.0,female,4.698251,0.0,Mutant,Unmethylated,non-codel,IDHmut-non-codel
1,TCGA-CS-4941,Brain Lower Grade Glioma,astrocytoma,G3,67.0,male,7.688047,1.0,WT,Methylated,non-codel,IDHwt
2,TCGA-CS-4942,Brain Lower Grade Glioma,astrocytoma,G3,44.0,female,43.861292,1.0,Mutant,Unmethylated,non-codel,IDHmut-non-codel


## 7. Selekcja kolumn i przygotowanie do mergu

Z 51 kolumn tabeli Ceccarelli wybieramy tylko te, które są nam potrzebne do analizy przeżycia i biostatystyki:

- **Identyfikator i stratyfikacja**: `Case`, `Study`
- **Dane kliniczne**: histologia, grade, wiek, płeć, Karnofsky Performance Score
- **Outcome (analiza przeżycia)**: czas przeżycia, status vitalny 
- **Biomarkery główne**: IDH, MGMT, 1p/19q codeletion, gotowy podtyp IDH/codel
- **Biomarkery dodatkowe** (opcjonalne do dyskusji): TERT, ATRX, mutation count

Przemianowujemy kolumny na **snake_case**

In [20]:
nazwy_kolumn = {
    # Identyfikatory i stratyfikacja
    'Case': 'patient_id',
    'Study': 'study',
    
    # Dane kliniczne
    'Histology': 'histology',
    'Grade': 'grade',
    'Age (years at diagnosis)': 'age_at_diagnosis',
    'Gender': 'gender',
    'Karnofsky Performance Score': 'kps',
    'Mutation Count': 'mutation_count',
    
    # Outcome - analiza przeżycia
    'Survival (months)': 'os_months',         # OS = Overall Survival
    'Vital status (1=dead)': 'os_event',      # event = zdarzenie (zgon)
    
    # Biomarkery główne
    'IDH status': 'idh_status',
    'MGMT promoter status': 'mgmt_status',
    '1p/19q codeletion': 'codel_1p19q',
    'IDH/codel subtype': 'idh_codel_subtype',
    
    # Biomarkery dodatkowe - do ewentualnej rozszerzonej analizy / dyskusji
    'TERT promoter status': 'tert_promoter_status',
    'ATRX status': 'atrx_status',
}

ceccarelli_clean = ceccarelli[list(nazwy_kolumn.keys())].rename(columns=nazwy_kolumn)

# Sprawdzenie wyniku
print(f"Kolumn przed: 51")
print(f"Kolumn po:    {len(ceccarelli_clean.columns)}")
print()
print("Nowe nazwy kolumn:")
for col in ceccarelli_clean.columns:
    print(f"  - {col}")

Kolumn przed: 51
Kolumn po:    16

Nowe nazwy kolumn:
  - patient_id
  - study
  - histology
  - grade
  - age_at_diagnosis
  - gender
  - kps
  - mutation_count
  - os_months
  - os_event
  - idh_status
  - mgmt_status
  - codel_1p19q
  - idh_codel_subtype
  - tert_promoter_status
  - atrx_status


In [21]:
# Podgląd pierwszych 5 wierszy z nowymi nazwami
ceccarelli_clean.head(5)

,patient_id,study,histology,grade,age_at_diagnosis,gender,kps,mutation_count,os_months,os_event,idh_status,mgmt_status,codel_1p19q,idh_codel_subtype,tert_promoter_status,atrx_status
0,TCGA-CS-4938,Brain Lower Grade Glioma,astrocytoma,G2,31.0,female,90.0,15.0,4.698251,0.0,Mutant,Unmethylated,non-codel,IDHmut-non-codel,WT,Mutant
1,TCGA-CS-4941,Brain Lower Grade Glioma,astrocytoma,G3,67.0,male,90.0,50.0,7.688047,1.0,WT,Methylated,non-codel,IDHwt,Mutant,WT
2,TCGA-CS-4942,Brain Lower Grade Glioma,astrocytoma,G3,44.0,female,90.0,24.0,43.861292,1.0,Mutant,Unmethylated,non-codel,IDHmut-non-codel,WT,Mutant
3,TCGA-CS-4943,Brain Lower Grade Glioma,astrocytoma,G3,37.0,male,50.0,30.0,18.135905,0.0,Mutant,Methylated,non-codel,IDHmut-non-codel,WT,Mutant
4,TCGA-CS-4944,Brain Lower Grade Glioma,astrocytoma,G2,50.0,male,90.0,20.0,10.612133,0.0,Mutant,Methylated,non-codel,IDHmut-non-codel,Mutant,WT


In [22]:
# info() pokazuje typy danych i liczbę niepustych wartości dla każdej kolumny
# pierwszy "data quality check"
ceccarelli_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1122 entries, 0 to 1121
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   patient_id            1122 non-null   str    
 1   study                 1122 non-null   str    
 2   histology             1047 non-null   str    
 3   grade                 1047 non-null   str    
 4   age_at_diagnosis      1047 non-null   float64
 5   gender                1047 non-null   str    
 6   kps                   697 non-null    float64
 7   mutation_count        820 non-null    float64
 8   os_months             1047 non-null   float64
 9   os_event              1046 non-null   float64
 10  idh_status            995 non-null    str    
 11  mgmt_status           932 non-null    str    
 12  codel_1p19q           1093 non-null   str    
 13  idh_codel_subtype     974 non-null    str    
 14  tert_promoter_status  329 non-null    str    
 15  atrx_status           820 non-nu

## 8. Walidacja zgodności identyfikatorów Ceccarelli vs cBioPortal

Zanim zmergujemy tabele, musimy sprawdzić, czy `patient_id` z Ceccarelli pasuje 1:1 do `patientId` z cBioPortal. Każde rozbieżności (pacjenci tylko w jednym źródle) trzeba zlokalizować i wyjaśnić.

Spodziewane: cBioPortal i Ceccarelli powinny mieć dokładnie te same 1122 patientów – obie listy pochodzą z tego samego TCGA Pan-Glioma cohort z roku 2015.


In [24]:
# Łączymy patientId z obu studiów cBioPortal w jeden zbiór
cbioportal_ids = (
    set(patients_per_study["GBM"]["patientId"])
    | set(patients_per_study["LGG"]["patientId"])
)

# To samo dla Ceccarelli
ceccarelli_ids = set(ceccarelli_clean["patient_id"])

# Porównanie - operatory na zbiorach
common = ceccarelli_ids & cbioportal_ids
only_ceccarelli = ceccarelli_ids - cbioportal_ids
only_cbioportal = cbioportal_ids - ceccarelli_ids

# Raport
print(f"Pacjentów w Ceccarelli:    {len(ceccarelli_ids):>5}")
print(f"Pacjentów w cBioPortal:    {len(cbioportal_ids):>5}")
print(f"Wspólnych (w obu źródłach): {len(common):>5}")
print(f"Tylko w Ceccarelli:         {len(only_ceccarelli):>5}")
print(f"Tylko w cBioPortal:         {len(only_cbioportal):>5}")

if only_ceccarelli:
    print(f"\nPrzykłady ID tylko w Ceccarelli ({len(only_ceccarelli)}):")
    for pid in sorted(only_ceccarelli)[:10]:
        print(f"  - {pid}")

if only_cbioportal:
    print(f"\nPrzykłady ID tylko w cBioPortal ({len(only_cbioportal)}):")
    for pid in sorted(only_cbioportal)[:10]:
        print(f"  - {pid}")

Pacjentów w Ceccarelli:     1122
Pacjentów w cBioPortal:     1122
Wspólnych (w obu źródłach):  1122
Tylko w Ceccarelli:             0
Tylko w cBioPortal:             0


## 9. Zapis tabeli klinicznej do `data/raw/`

Zapisujemy oczyszczoną tabelę Ceccarelli (16 kolumn, 1122 wiersze) jako CSV, żeby kolejne notebooki (Etap 2 – ETL) mogły ją wczytać bez ponownego parsowania pliku Excela.

Surowy plik xlsx zostaje w `data/raw/` jako referencja

In [25]:
# Ścieżka pliku wynikowego
ceccarelli_clean_path = RAW_DATA_DIR / "ceccarelli_clinical_clean.csv"

ceccarelli_clean.to_csv(ceccarelli_clean_path, index=False, encoding="utf-8")

# Sprawdzenie, że plik faktycznie powstał
if ceccarelli_clean_path.exists():
    rozmiar_kb = ceccarelli_clean_path.stat().st_size / 1024
    print(f"✓ Zapisano: {ceccarelli_clean_path}")
    print(f"  Rozmiar: {rozmiar_kb:.1f} KB")
    print(f"  Wiersze: {len(ceccarelli_clean)}")
    print(f"  Kolumny: {len(ceccarelli_clean.columns)}")
else:
    print("✗ Coś poszło nie tak - plik nie powstał")

✓ Zapisano: ..\data\raw\ceccarelli_clinical_clean.csv
  Rozmiar: 135.9 KB
  Wiersze: 1122
  Kolumny: 16
